# Course 6: Cleaning Data in Python (Revised)

This notebook covers diagnosing and fixing common data problems (data type, range, and uniqueness constraints), cleaning text and categorical data, resolving advanced data issues (uniformity, cross-field validation, completeness/missing data), and linking records across datasets using string similarity (`thefuzz` / `difflib`) and `recordlinkage`.

---
## Chapter 1: Common Data Problems

### 1.1 Data Type Constraints
* **Diagnosing Types**: Use `df.info()` and `df.dtypes` to inspect column types, and `df['col'].describe()` for summary statistics.
* **Numeric vs. Categorical**: Numeric codes representing discrete groups (e.g., `user_type` values `1`, `2`, `3`) should be converted to `'category'` via `.astype('category')` so `.describe()` returns frequency counts instead of meaningless averages.
* **Stripping Text from Numbers**: If a numeric column contains text suffixes (e.g., `'12 minutes'`), strip the text using `.str.strip('minutes')` and cast to integer with `.astype('int')`.
* **Verification**: Use `assert df['col'].dtype == 'int'` (or `np.issubdtype(df['col'].dtype, np.integer)`) to verify conversions programmatically.

### 1.2 Data Range Constraints
* Out-of-range values (e.g., tire sizes `> 27` or future dates `> today`) can be capped/hard-coded to valid boundaries using `.loc[condition, 'col'] = max_valid_value`.
* Convert string dates using `pd.to_datetime(df['col']).dt.date` and compare against `datetime.date.today()`.

### 1.3 Uniqueness Constraints (Duplicates)
* **Finding Duplicates**: `df.duplicated(subset=['col1', 'col2'], keep=False)` returns a boolean Series marking all duplicate rows (`keep='first'`, `keep='last'`, or `keep=False` to mark all occurrences).
* **Dropping Complete Duplicates**: `df.drop_duplicates(inplace=True)` removes rows that are 100% identical across all columns.
* **Collapsing Partial Duplicates**: Group by the unique identifier (`df.groupby('ride_id')`), aggregate conflicting columns using a rules dictionary (`.agg({'user_birth_year': 'min', 'duration_time': 'mean'})`), and restore a standard index with `.reset_index()`.

In [1]:
import pandas as pd
import numpy as np
import datetime as dt

# Load mock ride_sharing_new.csv dataset
ride_sharing = pd.read_csv('datasets/ride_sharing_new.csv')
print(ride_sharing.info())
print("\nUser type summary (as int):")
print(ride_sharing['user_type'].describe())

# Convert user_type to category and verify with assert
ride_sharing['user_type_cat'] = ride_sharing['user_type'].astype('category')
assert ride_sharing['user_type_cat'].dtype == 'category'
print("\nUser type summary (as category):")
print(ride_sharing['user_type_cat'].describe())

In [1]:
# Stripping 'minutes' string and converting duration to integer
ride_sharing['duration_trim'] = ride_sharing['duration'].str.strip('minutes').str.strip()
ride_sharing['duration_time'] = ride_sharing['duration_trim'].astype(int)

assert np.issubdtype(ride_sharing['duration_time'].dtype, np.integer)
print(ride_sharing[['duration', 'duration_trim', 'duration_time']].head(3))
print("Mean duration (minutes):", round(ride_sharing['duration_time'].mean(), 2))

In [1]:
# Enforcing Data Range Constraints on tire_sizes and ride_date
ride_sharing['tire_sizes'] = ride_sharing['tire_sizes'].astype(int)

# Cap values above 27 to 27 using .loc
ride_sharing.loc[ride_sharing['tire_sizes'] > 27, 'tire_sizes'] = 27
ride_sharing['tire_sizes'] = ride_sharing['tire_sizes'].astype('category')
print("Capped tire_sizes value counts:")
print(ride_sharing['tire_sizes'].value_counts())

# Convert ride_date to date and cap future dates to today
ride_sharing['ride_dt'] = pd.to_datetime(ride_sharing['ride_date']).dt.date
today = dt.date.today()
ride_sharing.loc[ride_sharing['ride_dt'] > today, 'ride_dt'] = today
print("Max ride_dt after capping:", ride_sharing['ride_dt'].max())

In [1]:
# Finding and resolving complete & partial duplicates on ride_id
duplicates = ride_sharing.duplicated(subset='ride_id', keep=False)
duplicated_rides = ride_sharing[duplicates].sort_values('ride_id', ascending=True)
print("Duplicated rides before cleaning:")
print(duplicated_rides[['ride_id', 'duration_time', 'user_birth_year']])

# 1. Drop complete duplicates
ride_dup = ride_sharing.drop_duplicates()

# 2. Define aggregation dictionary and collapse partial duplicates by ride_id
statistics = {'user_birth_year': 'min', 'duration_time': 'mean'}
ride_unique = ride_dup.groupby('ride_id').agg(statistics).reset_index()

# 3. Verify 0 duplicates remain
duplicates_after = ride_unique.duplicated(subset='ride_id', keep=False)
assert ride_unique[duplicates_after].shape[0] == 0
print("\nCleaned unique rides count:", ride_unique.shape[0])

### Chapter 1: Key Takeaways & Actionable Step-by-Step Process

#### Key Takeaways
* Data types dictate which operations are valid—strip text units and cast strings to `int`/`float`/`category`/`datetime` immediately after loading.
* Use `.loc[condition, col] = limit` to cap out-of-range numbers or future timestamps without dropping entire records.
* Resolve duplicate records in two stages: `.drop_duplicates()` for exact row copies, followed by `.groupby(id).agg(dict).reset_index()` for partial duplicates.

#### Step-by-Step Process to Act On
1. **Audit Schema**: Run `df.info()` and `.describe()` to spot numeric columns stored as `object` or categories stored as `int`.
2. **Cast & Assert**: Clean strings with `.str.strip()`, convert with `.astype()` or `pd.to_datetime()`, and lock in with `assert`.
3. **Cap Out-of-Range Values**: Filter invalid values with `.loc[]` and overwrite them with domain-valid ceilings/floors.
4. **Deduplicate**: Run `.duplicated(subset=..., keep=False)` to inspect duplicates, drop exact copies, and aggregate partial duplicates via `.groupby().agg()`.

---
### Chapter 1: Glossary & Official Documentation

| Item | Category | Explanation | Arguments Accepted | Official Documentation |
| :--- | :--- | :--- | :--- | :--- |
| `Series.astype()` | `pandas` Method | Casts a pandas object to a specified dtype (`int`, `float`, `'category'`, `str`). | `dtype`, `copy=True`, `errors='raise'` | [Pandas `astype` Docs](https://pandas.pydata.org/docs/reference/api/pandas.Series.astype.html) |
| `Series.str.strip()` | `pandas` Method | Removes leading and trailing characters (spaces by default, or specified chars). | `to_strip=None` | [Pandas `str.strip` Docs](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.strip.html) |
| `pd.to_datetime()` | `pandas` Function | Converts argument to `datetime` type. | `arg`, `errors='raise'`, `format=None` | [Pandas `to_datetime` Docs](https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html) |
| `DataFrame.loc[]` | `pandas` Indexer | Accesses a group of rows and columns by label(s) or a boolean array. | `[row_indexer, col_indexer]` | [Pandas `loc` Docs](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.loc.html) |
| `DataFrame.duplicated()` | `pandas` Method | Returns boolean Series denoting duplicate rows. | `subset=None`, `keep='first'` (`'first'`, `'last'`, `False`) | [Pandas `duplicated` Docs](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html) |
| `DataFrame.drop_duplicates()` | `pandas` Method | Returns DataFrame with duplicate rows removed. | `subset=None`, `keep='first'`, `inplace=False` | [Pandas `drop_duplicates` Docs](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html) |
| `DataFrame.groupby()` | `pandas` Method | Groups DataFrame using a mapper or by a Series of columns. | `by=None`, `axis=0`, `as_index=True` | [Pandas `groupby` Docs](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html) |
| `GroupBy.agg()` | `pandas` Method | Aggregates using one or more operations over the specified axis. | `func` (function, str, list, or dict) | [Pandas `agg` Docs](https://pandas.pydata.org/docs/reference/api/pandas.core.groupby.DataFrameGroupBy.agg.html) |

---
## Chapter 2: Text and Categorical Data Problems

### 2.1 Membership Constraints (Anti-Joins on Categories)
* Categorical variables belong to a predefined finite set of valid values.
* **Finding Inconsistent Categories**: Use Python `set.difference()` between the column's unique values and a master reference table:
  `inconsistent_cats = set(df['col']).difference(categories['col'])`
* **Filtering & Dropping with `~` and `.isin()`**:
  * `inconsistent_rows = df['col'].isin(inconsistent_cats)` creates a boolean mask.
  * `consistent_df = df[~inconsistent_rows]` uses the bitwise negation operator `~` ("not") to drop inconsistent rows.

### 2.2 Categorical Variables (Value Consistency & Collapsing)
* **Formatting Consistency**: Normalize case via `.str.lower()` / `.str.upper()` and strip whitespace via `.str.strip()`.
* **Collapsing / Binning Continuous Data**:
  * `pd.qcut(series, q=3, labels=group_names)`: Bins data into quantile-based buckets.
  * `pd.cut(series, bins=ranges, labels=group_names)`: Bins data into custom numeric intervals (e.g., `[0, 60, 180, np.inf]`).
* **Remapping Categories**: Use `.replace(mapping_dict)` to collapse granular categories (e.g., days of the week $\rightarrow$ `'weekday'` / `'weekend'`).

### 2.3 Cleaning Text Data
* Strip honorifics/substrings using `.str.replace("Dr.", "", regex=False)` and verify removal with `assert df['col'].str.contains('Ms.|Mr.|Miss|Dr.').any() == False`.
* Filter rows based on character length using `.str.len() > threshold`.

In [1]:
# Membership constraints: Finding and dropping inconsistent categories in airlines_final.csv
import pandas as pd
import numpy as np

airlines = pd.read_csv('datasets/airlines_final.csv')
categories = pd.read_csv('datasets/categories.csv')

print("Unique cleanliness values in airlines:", airlines['cleanliness'].unique())

# Find inconsistent categories using set.difference()
cat_clean = set(airlines['cleanliness']).difference(categories['cleanliness'])
print("Inconsistent cleanliness category found:", cat_clean)

# Isolate inconsistent rows with .isin() and drop them with ~
cat_clean_rows = airlines['cleanliness'].isin(cat_clean)
print("\nInconsistent row(s):")
print(airlines[cat_clean_rows][['id', 'full_name', 'cleanliness']])

consistent_airlines = airlines[~cat_clean_rows]
print("Rows remaining after dropping inconsistent categories:", len(consistent_airlines))

In [1]:
# Value consistency, pd.cut() binning, and .replace() category mapping
airlines['dest_region'] = airlines['dest_region'].str.lower()
airlines['dest_region'] = airlines['dest_region'].replace({'eur': 'europe'})
airlines['dest_size'] = airlines['dest_size'].str.strip()

print("Cleaned dest_region:", airlines['dest_region'].unique())
print("Cleaned dest_size:", airlines['dest_size'].unique())

# Bin wait_min into wait_type categories using pd.cut()
label_ranges = [0, 60, 180, np.inf]
label_names = ['short', 'medium', 'long']
airlines['wait_type'] = pd.cut(airlines['wait_min'], bins=label_ranges, labels=label_names)

# Map days to weekday/weekend
mappings = {
    'Monday': 'weekday', 'Tuesday': 'weekday', 'Wednesday': 'weekday',
    'Thursday': 'weekday', 'Friday': 'weekday',
    'Saturday': 'weekend', 'Sunday': 'weekend'
}
airlines['day_week'] = airlines['day'].replace(mappings)
print("\nSample binned and mapped columns:")
print(airlines[['day', 'day_week', 'wait_min', 'wait_type']].head())

In [1]:
# Cleaning text data: Removing honorifics and filtering by survey_response length
for title in ["Dr.", "Mr.", "Miss", "Ms."]:
    airlines['full_name'] = airlines['full_name'].str.replace(title, "", regex=False).str.strip()

# Assert that full_name has no honorifics remaining
assert airlines['full_name'].str.contains('Ms.|Mr.|Miss|Dr.').any() == False
print("Cleaned names:", airlines['full_name'].tolist())

# Filter survey responses longer than 40 characters
resp_length = airlines['survey_response'].str.len()
airlines_survey = airlines[resp_length > 40]
assert airlines_survey['survey_response'].str.len().min() > 40
print("\nFiltered detailed survey responses:")
print(airlines_survey[['full_name', 'survey_response']])

### Chapter 2: Key Takeaways & Actionable Step-by-Step Process

#### Key Takeaways
* `set(df['col']).difference(valid_set)` combined with `.isin()` and `~` provides a fast, set-theoretic anti-join to eliminate invalid categorical entries.
* Whitespace and casing mismatches (`'  Large'` vs `'Large'`, `'EAST US'` vs `'East US'`) silently fragment categorical groupings until normalized with `.str.strip()` and `.str.lower()`.
* `pd.cut()` and `.replace(mapping_dict)` reduce high-cardinality variables into clean analytical cohorts.

#### Step-by-Step Process to Act On
1. **Normalize Strings First**: Apply `.str.strip()` and `.str.lower()` to all categorical/text columns.
2. **Check Membership**: Diff unique column values against your ground-truth lookup table using `set().difference()` and filter out invalid rows via `df[~df['col'].isin(invalid)]`.
3. **Collapse & Bin**: Map synonyms or group days using `.replace(dict)` and discretize continuous variables using `pd.cut()`.
4. **Clean Free Text**: Strip prefixes/titles with `.str.replace()` and validate with `assert not df['col'].str.contains(pattern).any()`.

---
### Chapter 2: Glossary & Official Documentation

| Item | Category | Explanation | Arguments Accepted | Official Documentation |
| :--- | :--- | :--- | :--- | :--- |
| `set.difference()` | Python `set` Method | Returns a new set with elements in the set that are not in the `others`. | `*others` (iterables) | [Python `set.difference` Docs](https://docs.python.org/3/library/stdtypes.html#frozenset.difference) |
| `Series.isin()` | `pandas` Method | Whether elements in Series are contained in `values` (returns boolean Series). | `values` (set or list-like) | [Pandas `isin` Docs](https://pandas.pydata.org/docs/reference/api/pandas.Series.isin.html) |
| `Series.unique()` | `pandas` Method | Returns unique values of Series object in order of appearance. | None | [Pandas `unique` Docs](https://pandas.pydata.org/docs/reference/api/pandas.Series.unique.html) |
| `pd.cut()` | `pandas` Function | Bins values into discrete intervals. | `x`, `bins`, `right=True`, `labels=None` | [Pandas `cut` Docs](https://pandas.pydata.org/docs/reference/api/pandas.cut.html) |
| `pd.qcut()` | `pandas` Function | Quantile-based discretization function. | `x`, `q`, `labels=None` | [Pandas `qcut` Docs](https://pandas.pydata.org/docs/reference/api/pandas.qcut.html) |
| `Series.replace()` | `pandas` Method | Replaces values given in `to_replace` with `value` (or via mapping dict). | `to_replace=None`, `value=NoDefault` | [Pandas `replace` Docs](https://pandas.pydata.org/docs/reference/api/pandas.Series.replace.html) |
| `Series.str.contains()` | `pandas` Method | Tests if pattern or regex is contained within a string of a Series. | `pat`, `case=True`, `flags=0`, `na=None`, `regex=True` | [Pandas `str.contains` Docs](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.contains.html) |
| `Series.str.len()` | `pandas` Method | Computes the length of each string element in the Series. | None | [Pandas `str.len` Docs](https://pandas.pydata.org/docs/reference/api/pandas.Series.str.len.html) |

---
## Chapter 3: Advanced Data Problems

### 3.1 Uniformity (Units, Currencies & Ambiguous Dates)
* Columns must share a single unit of measurement (e.g., converting `'euro'` account amounts to `'dollar'` by multiplying by the exchange rate via `.loc[mask, 'acct_amount']`).
* Mixed or corrupt date formats are standardized using `pd.to_datetime(df['col'], errors='coerce')`, which converts unparseable strings into `NaT` (missing value) instead of raising an error. Extract components using `.dt.strftime('%Y')` or `.dt.year`.

### 3.2 Cross-Field Validation
* Verifies data integrity by comparing multiple columns against each other:
  * **Sum Check**: Checking if `fund_A + fund_B + fund_C + fund_D == inv_amount`.
  * **Age Check**: Checking if `today.year - birth_date.dt.year == age`.
* Isolate consistent rows (`df[equ_mask]`) from inconsistent rows (`df[~equ_mask]`) for downstream remediation.

### 3.3 Completeness & Missing Data (`isna`, `dropna`, `fillna`)
* **Types of Missingness**:
  * **MCAR** (Missing Completely at Random): No systematic relationship with any values.
  * **MAR** (Missing at Random): Missingness depends on another observed variable (e.g., `inv_amount` missing only for younger `age` rows).
  * **MNAR** (Missing Not at Random): Missingness depends on the unobserved value itself.
* **Handling Missing Values**:
  * Count missing values per column: `df.isna().sum()`.
  * Drop rows missing critical identifiers: `df.dropna(subset=['cust_id'])`.
  * Impute missing values using statistical/domain formulas: `df.fillna({'acct_amount': acct_imp})`.

In [1]:
# Uniformity & Cross-Field Validation on banking_dirty.csv
import pandas as pd
import datetime as dt

banking = pd.read_csv('datasets/banking_dirty.csv')

# 1. Currency Uniformity: Convert euro to dollar
acct_eu = banking['acct_cur'] == 'euro'
banking.loc[acct_eu, 'acct_amount'] = banking.loc[acct_eu, 'acct_amount'] * 1.1
banking.loc[acct_eu, 'acct_cur'] = 'dollar'
assert (banking['acct_cur'].unique() == ['dollar']).all()

# 2. Date Uniformity with errors='coerce'
banking['account_opened'] = pd.to_datetime(banking['account_opened'], format='mixed', errors='coerce')
banking['acct_year'] = banking['account_opened'].dt.strftime('%Y')
print("Standardized account_opened & acct_year:")
print(banking[['cust_id', 'account_opened', 'acct_year']].head())

# 3. Cross-Field Validation: Birth Date vs Age
banking['birth_date'] = pd.to_datetime(banking['birth_date'])
today = dt.date(2026, 10, 31)
ages_manual = today.year - banking['birth_date'].dt.year
age_equ = banking['age'] == ages_manual

consistent_ages = banking[age_equ]
inconsistent_ages = banking[~age_equ]
print("\nNumber of inconsistent ages found:", inconsistent_ages.shape[0])

In [1]:
# Completeness: Diagnosing, Dropping, and Imputing Missing Data
print("Missing values per column before cleaning:")
print(banking.isna().sum())

# Isolate missing and non-missing values of inv_amount to inspect pattern
missing_investors = banking[banking['inv_amount'].isna()]
investors = banking[~banking['inv_amount'].isna()]
print("\nMean age when inv_amount is missing:", missing_investors['age'].mean())
print("Mean age when inv_amount is present:", investors['age'].mean())

# Drop rows missing cust_id and impute missing acct_amount as 5x inv_amount
banking_fullid = banking.dropna(subset=['cust_id']).copy()
acct_imp = banking_fullid['inv_amount'] * 5
banking_imputed = banking_fullid.fillna({'acct_amount': acct_imp})

print("\nMissing values after dropping missing cust_id and imputing acct_amount:")
print(banking_imputed[['cust_id', 'acct_amount', 'inv_amount']].isna().sum())

### Chapter 3: Key Takeaways & Actionable Step-by-Step Process

#### Key Takeaways
* Unit/currency conversions must update **both** the numeric value column and the unit label column simultaneously.
* Passing `errors='coerce'` to `pd.to_datetime()` prevents corrupt date strings from halting execution by turning them into `NaT`.
* Cross-field validation catches logical contradictions (like `sum(sub_funds) != total_investment` or `2026 - birth_year != age`) that single-column checks miss.
* Comparing `.describe()` across `df[col.isna()]` and `df[~col.isna()]` reveals whether data is missing at random or tied to another variable like `age`.

#### Step-by-Step Process to Act On
1. **Unify Units & Dates**: Standardize currencies/units using `.loc[]` and parse dates with `pd.to_datetime(..., errors='coerce')`.
2. **Run Cross-Field Checks**: Compare derived totals/ages against reported columns and isolate mismatches with `df[~mask]`.
3. **Diagnose Missingness**: Run `df.isna().sum()` and compare summary stats between missing and complete subsets.
4. **Drop or Impute**: Use `.dropna(subset=[...])` for unrecoverable key columns and `.fillna({...})` for estimable numeric columns.

---
### Chapter 3: Glossary & Official Documentation

| Item | Category | Explanation | Arguments Accepted | Official Documentation |
| :--- | :--- | :--- | :--- | :--- |
| `Series.dt.strftime()` | `pandas` Method | Converts datetime Series to formatted strings using standard `strftime` codes. | `date_format` (`str`, e.g., `'%Y-%m-%d'`) | [Pandas `dt.strftime` Docs](https://pandas.pydata.org/docs/reference/api/pandas.Series.dt.strftime.html) |
| `DataFrame.isna()` | `pandas` Method | Detects missing values (`NaN` / `None` / `NaT`), returning a boolean DataFrame. | None | [Pandas `isna` Docs](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html) |
| `DataFrame.dropna()` | `pandas` Method | Removes missing values along a specified axis or column subset. | `axis=0`, `how='any'`, `subset=None`, `inplace=False` | [Pandas `dropna` Docs](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html) |
| `DataFrame.fillna()` | `pandas` Method | Fills `NA`/`NaN` values using the specified value, dict, Series, or method. | `value=None`, `axis=None`, `inplace=False` | [Pandas `fillna` Docs](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.fillna.html) |
| `msno.matrix()` | `missingno` Function | Visualizes missing data density and co-occurrence patterns across rows. | `df`, `filter=None`, `n=0`, `p=0`, `sort=None` | [Missingno GitHub Docs](https://github.com/ResidentMario/missingno) |

---
## Chapter 4: Record Linkage

### 4.1 Minimum Edit Distance & Fuzzy String Matching (`thefuzz` / `difflib`)
* **Minimum Edit Distance**: The minimum number of single-character operations (insertion, deletion, substitution, transposition) required to transform one string into another (e.g., Levenshtein distance).
* **Using `thefuzz` (`fuzz` and `process`)**:
  * `fuzz.WRatio('Reeding', 'Reading')`: Returns a similarity score from `0` to `100`.
  * `process.extract(target_str, choices_series, limit=n)`: Compares `target_str` against every string in `choices_series` and returns a list of `(match_string, score, index)` tuples.
  * Iterate over matches and remap typos where `match[1] >= 80` (cutoff threshold).

### 4.2 Record Linkage Across DataFrames (`recordlinkage`)
* When joining two messy tables that lack a shared unique key (`restaurants` and `restaurants_new`), comparing every row against every other row ($N \times M$ Cartesian product) is computationally expensive.
* **4-Step Record Linkage Pipeline**:
  1. **Blocking (`recordlinkage.Index()`)**: Create an indexer (`indexer = recordlinkage.Index()`), block on a reliable shared column (`indexer.block('city')`), and generate candidate index pairs (`pairs = indexer.index(df_A, df_B)`).
  2. **Comparison Rules (`recordlinkage.Compare()`)**: Define exact matches (`comp_cl.exact('city', 'city', label='city')`) and string similarity matches (`comp_cl.string('rest_name', 'rest_name', threshold=0.8, label='name')`).
  3. **Compute Scores**: `potential_matches = comp_cl.compute(pairs, df_A, df_B)` returns a MultiIndex DataFrame of `1`s (match) and `0`s (non-match).
  4. **Filter & Link**: Isolate true matches via `matches = potential_matches[potential_matches.sum(axis=1) >= 2]`, extract duplicate indices from `df_B` via `matches.index.get_level_values(1)`, drop them from `df_B`, and append the truly new records to `df_A` via `pd.concat([df_A, df_B_new])`.

In [1]:
# Collapsing typo categories using string similarity (compatible with both thefuzz and standard library difflib)
import pandas as pd
from difflib import SequenceMatcher

try:
    from thefuzz import process
except ImportError:
    # Fallback wrapper matching thefuzz.process.extract signature (match_str, score_0_100, idx)
    class process:
        @staticmethod
        def extract(query, choices, limit=5):
            results = []
            for idx, val in enumerate(choices):
                score = int(round(SequenceMatcher(None, str(query).lower(), str(val).lower()).ratio() * 100))
                results.append((val, score, idx))
            results.sort(key=lambda x: x[1], reverse=True)
            return results[:limit]

restaurants = pd.read_csv('datasets/restaurants_L2.csv')
print("Unique types before fuzzy remapping:", restaurants['type'].unique())

categories = ['asian', 'american', 'italian', 'southern']
for cuisine in categories:
    matches = process.extract(cuisine, restaurants['type'], limit=len(restaurants['type']))
    for match in matches:
        if match[1] >= 80:
            restaurants.loc[restaurants['type'] == match[0], 'type'] = cuisine

print("Unique types after fuzzy remapping (cutoff >= 80):", restaurants['type'].unique())

In [1]:
# Full Record Linkage Workflow: Blocking, Comparing, Scoring, and Linking DataFrames
import pandas as pd
from difflib import SequenceMatcher

restaurants_A = pd.read_csv('datasets/restaurants_L2.csv')
restaurants_B = pd.read_csv('datasets/restaurants_L2_dirty.csv')

# 1. Blocking on 'city' to generate candidate index pairs
pairs = []
for idx_a, row_a in restaurants_A.iterrows():
    for idx_b, row_b in restaurants_B.iterrows():
        if row_a['city'] == row_b['city']:
            pairs.append((idx_a, idx_b))
pairs_index = pd.MultiIndex.from_tuples(pairs, names=['idx_A', 'idx_B'])

# 2. Compute exact matches on ('city', 'cuisine_type') and string similarity (>= 0.8) on 'rest_name'
records = []
for idx_a, idx_b in pairs_index:
    city_match = int(restaurants_A.loc[idx_a, 'city'] == restaurants_B.loc[idx_b, 'city'])
    cuisine_match = int(restaurants_A.loc[idx_a, 'type'] == restaurants_B.loc[idx_b, 'type'])
    name_sim = SequenceMatcher(None, restaurants_A.loc[idx_a, 'rest_name'], restaurants_B.loc[idx_b, 'rest_name']).ratio()
    name_match = int(name_sim >= 0.8)
    records.append({'city': city_match, 'cuisine_type': cuisine_match, 'name': name_match})

potential_matches = pd.DataFrame(records, index=pairs_index)
print("Potential Matches Score Matrix:")
print(potential_matches)

# 3. Isolate probable matches (score >= 3) and link DataFrames without duplicates
matches = potential_matches[potential_matches.sum(axis=1) >= 3]
print("\nConfirmed Duplicate Pairs (sum >= 3):")
print(matches)

duplicate_rows_B = matches.index.get_level_values(1)
restaurants_B_new = restaurants_B[~restaurants_B.index.isin(duplicate_rows_B)]
full_restaurants = pd.concat([restaurants_A, restaurants_B_new], ignore_index=True)
print(f"\nLinked DataFrame rows: {len(restaurants_A)} (A) + {len(restaurants_B_new)} (new from B) = {len(full_restaurants)} total.")

### Chapter 4: Key Takeaways & Actionable Step-by-Step Process

#### Key Takeaways
* Fuzzy string matching (`thefuzz.process.extract`) automates typo correction when a column contains dozens of misspellings of known categories (`'italiian'`, `'americen'`).
* Record linkage joins datasets that lack exact foreign keys by blocking on a clean attribute (e.g., `'city'`), scoring candidate pairs across exact and fuzzy fields, and deduplicating before `pd.concat()`.

#### Step-by-Step Process to Act On
1. **Remap Typos with `thefuzz`**: Loop through canonical categories, run `process.extract(cat, df['col'])`, and overwrite values scoring `>= 80`.
2. **Block Candidate Pairs**: Initialize `indexer = recordlinkage.Index()`, call `indexer.block('city')`, and generate `pairs = indexer.index(df_A, df_B)`.
3. **Score Pairs**: Configure `recordlinkage.Compare()` with `.exact()` and `.string(..., threshold=0.8)` rules, then call `.compute(pairs, df_A, df_B)`.
4. **Filter & Concatenate**: Filter `potential_matches[potential_matches.sum(axis=1) >= threshold]`, drop matched indices from `df_B` using `.index.get_level_values(1)`, and combine with `pd.concat()`.

---
### Chapter 4: Glossary & Official Documentation

| Item | Category | Explanation | Arguments Accepted | Official Documentation |
| :--- | :--- | :--- | :--- | :--- |
| `fuzz.WRatio()` | `thefuzz` Function | Calculates weighted Levenshtein similarity ratio between two strings (`0`–`100`). | `s1` (`str`), `s2` (`str`), `force_ascii=True`, `full_process=True` | [TheFuzz GitHub Docs](https://github.com/seatgeek/thefuzz) |
| `process.extract()` | `thefuzz` Function | Finds best fuzzy matches for a query string within a collection of choices. | `query`, `choices`, `processor=...`, `scorer=...`, `limit=5` | [TheFuzz `process` Docs](https://github.com/seatgeek/thefuzz#process) |
| `recordlinkage.Index()` | `recordlinkage` Class | Class for generating candidate record pairs using blocking or full indexing. | `blocks=None` | [RecordLinkage `Index` Docs](https://recordlinkage.readthedocs.io/en/latest/ref-index.html) |
| `recordlinkage.Compare()` | `recordlinkage` Class | Class defining exact, string, numeric, and geographic comparison rules between pairs. | `features=None`, `n_jobs=1` | [RecordLinkage `Compare` Docs](https://recordlinkage.readthedocs.io/en/latest/ref-compare.html) |
| `MultiIndex.get_level_values()` | `pandas` Method | Returns vector of label values for requested level in a MultiIndex. | `level` (`int` or `str`) | [Pandas `get_level_values` Docs](https://pandas.pydata.org/docs/reference/api/pandas.MultiIndex.get_level_values.html) |
| `pd.concat()` | `pandas` Function | Concatenates pandas objects along a particular axis (`0` for rows, `1` for columns). | `objs`, `axis=0`, `ignore_index=False` | [Pandas `concat` Docs](https://pandas.pydata.org/docs/reference/api/pandas.concat.html) |